In [27]:
import pandas as pd 
import matplotlib.pyplot as plt

In [28]:
file = "data/ecis_simulated_pilot (1)/pilot_sim_001_C1_R1.csv"
df = pd.read_csv(file)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/ecis_simulated_pilot (1)/pilot_sim_001_C1_R1.csv'

In [ ]:
import os 
os.getcwd()

In [ ]:
os.listdir()

In [ ]:
import pandas as pd

file = "../data/ecis_simulated_pilot (1)/pilot_sim_001_C1_R1.csv"
df = pd.read_csv(file)

df.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(df["Time_min"], df["Impedance_Magnitude"])

plt.xlabel("Time (min)")
plt.ylabel("Impedance")
plt.title("ECIS Signal")

plt.show()

In [ ]:
# multiple frequences are shows filter one out 
df_500 = df[df["Frequency_Hz"] == 500]

plt.figure(figsize=(8,5))
plt.plot(df_500["Time_min"], df_500["Impedance_Magnitude"])

plt.xlabel("Time (min)")
plt.ylabel("Impedance")
plt.title("ECIS Signal (500 Hz)")

plt.show() # SIGNAL SHOWS ONE FREQUENCY (500HZ) ONE CLEAN ECIS CURVE 
#STARTS AT ABOUT 865 GUADUALLY INCREASES TO ABOUT 995 SMALL NOISE FLUCTUATIONS

In [ ]:
# NORMALIZATION

# Get the first impedance value at 500 Hz to use as the baseline
baseline = df_500["Impedance_Magnitude"].iloc[0]

# Create a new column with impedance normalized to the baseline
# Each value is divided by the first value so the curve starts near 1.0
df_500["Normalized"] = df_500["Impedance_Magnitude"] / baseline

# Create a plot of normalized impedance over time
plt.figure(figsize=(8,5))
plt.plot(df_500["Time_min"], df_500["Normalized"])

# Label the axes
plt.xlabel("Time (min)")
plt.ylabel("Normalized Impedance")

# Add a title
plt.title("Normalized ECIS Signal (500 Hz)")

# Display the plot
plt.show()

In [ ]:
#replicating comparisons 
import pandas as pd
import matplotlib.pyplot as plt

files = [
    "../data/ecis_simulated_pilot (1)/pilot_sim_001_C1_R1.csv",
    "../data/ecis_simulated_pilot (1)/pilot_sim_001_C1_R2.csv",
    "../data/ecis_simulated_pilot (1)/pilot_sim_001_C1_R3.csv"
]

plt.figure(figsize=(8,5))

for file in files:
    df = pd.read_csv(file)
    df_500 = df[df["Frequency_Hz"] == 500].copy()
    
    baseline = df_500["Impedance_Magnitude"].iloc[0]
    norm = df_500["Impedance_Magnitude"] / baseline
    
    plt.plot(df_500["Time_min"], norm, alpha=0.8)

plt.xlabel("Time (min)")
plt.ylabel("Normalized Impedance")
plt.title("Replicate Comparison (C1)")

plt.show()
# comparing R1. R2, R3 together 

In [ ]:
import numpy as np

replicates = []

for file in files:
    df = pd.read_csv(file)
    df_500 = df[df["Frequency_Hz"] == 500].copy()

    baseline = df_500["Impedance_Magnitude"].iloc[0]
    norm = df_500["Impedance_Magnitude"] / baseline

    replicates.append(norm.values)

replicates = np.array(replicates)

mean_signal = replicates.mean(axis=0)
std_signal = replicates.std(axis=0)

time = df_500["Time_min"]

plt.figure(figsize=(8,5))
plt.plot(time, mean_signal, label="Mean")

plt.fill_between(
    time,
    mean_signal - std_signal,
    mean_signal + std_signal,
    alpha=0.3
)

plt.xlabel("Time (min)")
plt.ylabel("Normalized Impedance")
plt.title("Mean ± Variance (C1)")
plt.legend()

plt.show()


## Pilot ECIS Data Analysis

The pilot ECIS datasets were successfully loaded and processed using Python in a JupyterLab environment. Signals were filtered to a single measurement frequency of 500 Hz, and impedance values were normalized to baseline impedance at the start of the experiment to enable direct comparison across wells.

Replicate comparison across R1–R3 wells showed consistent impedance trajectories with relatively low variance between replicates, suggesting stable cell seeding and reliable signal acquisition during the measurement period.

The preprocessing pipeline, including data loading, frequency filtering, baseline normalization, and replicate visualization, is functioning correctly. Based on these results, the pilot datasets appear suitable for subsequent analysis of experimental conditions and treatment effects on cell behavior.


## Nest step: Quanatative Analysis

After confirming that the pilot ECIS preprocessing pipeline is functioning correctly,
the next stage is to quantify the impedance response across replicates. This includes
calculating the mean normalized impedance trajectory, measuring replicate variability,
and extracting summary metrics such as slope, maximum impedance, and plateau behavior.
These metrics will support later comparisons between experimental conditions and drug-
timing strategies.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Group the 500 Hz dataframe by time point
# This combines replicate measurements taken at the same time
grouped = df_500.groupby("Time_min")["Normalized"]

# Calculate the average normalized impedance at each time point
mean_impedance = grouped.mean()

# Calculate the standard deviation at each time point
# This measures how much the replicates vary from one another
std_impedance = grouped.std()

# Save the time values from the grouped result
time = mean_impedance.index

# Plot the mean normalized impedance curve
plt.figure(figsize=(8,5))
plt.plot(time, mean_impedance, label="Mean normalized impedance")

# Add a shaded band for mean ± standard deviation
plt.fill_between(
    time,
    mean_impedance - std_impedance,
    mean_impedance + std_impedance,
    alpha=0.3,
    label="± SD"
)

# Label the axes and add a title
plt.xlabel("Time (min)")
plt.ylabel("Normalized Impedance")
plt.title("ECIS Mean Response Across Replicates")

# Show the legend and plot
plt.legend()
plt.show()


In [ ]:
# Add a column to label the experimental condition

df_500["Condition"] = "Control"

In [ ]:
# FEATURE EXTRACTION
# Find the maximum normalized impedance value
max_impedance = df_500["Normalized"].max()

# Find the time at which the maximum impedance occurs
time_of_max = df_500.loc[df_500["Normalized"].idxmax(), "Time_min"]

# Estimate the overall growth rate using the first and last points
slope = (
    df_500["Normalized"].iloc[-1] - df_500["Normalized"].iloc[0]
) / (
    df_500["Time_min"].iloc[-1] - df_500["Time_min"].iloc[0]
)

print("Maximum normalized impedance:", max_impedance)
print("Time of maximum impedance:", time_of_max)
print("Approximate growth slope:", slope)

In [ ]:
# STORE FEATURES IN A TABLE (this will scale later)

results = pd.DataFrame({
    "Condition": ["Control"],
    "Max_Impedance": [max_impedance],
    "Time_of_Max": [time_of_max],
    "Slope": [slope]
})

results

## Feature Extraction Analysis

Quantitative features were extracted from the normalized ECIS signal to summarize cell behavior over time.

The maximum normalized impedance was approximately **1.154**, occurring at **4170 minutes**, indicating that cell coverage steadily increased throughout the duration of the experiment and reached its peak near the end of the observation period.

The estimated growth slope was **3.19 × 10⁻⁵**, reflecting a gradual and consistent increase in impedance over time. This suggests stable cell attachment and proliferation rather than rapid or abrupt changes in behavior.

Overall, the ECIS signal exhibits a smooth upward trajectory with low variability, supporting the conclusion that the experimental conditions produced reliable and steady cell growth dynamics.

These extracted features provide a quantitative foundation for future comparisons across experimental conditions and drug-timing strategies.

## Condition Comparison (Preliminary)

At this stage, analysis has been performed on a single pilot dataset at 500 Hz. While no experimental conditions are being compared yet, 
the extracted features and summary statistics establish a baseline reference for future comparisons.

Subsequent analyses will involve comparing these metrics across different experimental conditions, such as varying drug timing strategies, 
to evaluate their impact on cell behavior and impedance dynamics.

In [ ]:
# Plot max impedance (will be more useful with multiple conditions)

plt.figure(figsize=(6,4))
plt.bar(results["Condition"], results["Max_Impedance"])

plt.xlabel("Condition")
plt.ylabel("Max Normalized Impedance")
plt.title("Max Impedance by Condition")

plt.show()

## Condition Comparison (Preliminary)

A structured comparison framework was established to enable evaluation of different experimental conditions.

Although only a single pilot condition ("Control") is currently available, key features including maximum impedance, time to peak, and growth slope were organized into a results table. This structure will allow for direct comparison as additional conditions are introduced.

This approach ensures that future analyses of drug timing strategies can be performed consistently and quantitatively across multiple experimental groups.

## Progress Note (March 17)

Completed normalization, replicate analysis, mean ± standard deviation visualization, and feature extraction (maximum impedance, time to peak, and slope).

Established a structured framework for condition comparison to support future analysis of drug timing effects.